In [ ]:
# セル 1: 必要なライブラリのインポート
import glob
import os

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# from sklearn.preprocessing import StandardScaler # 今回はターゲットの正規化は一旦保留
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import img_to_array, load_img


2025-05-14 12:29:17.840566: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-14 12:29:17.860235: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747193357.884314 3110214 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747193357.891400 3110214 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747193357.908930 3110214 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# 再現性のための乱数シード設定
np.random.seed(42)
tf.random.set_seed(42)

# セル 2: 設定値
IMG_BASE_PATH = "./data/20250425_data/Material_d/"  # 画像ファイルが格納されているディレクトリ (例: SingleI_0_0.bmp)
CSV_BASE_PATH = "./data/20250425_data/TorqData/"  # CSVファイルが格納されているディレクトリ (例: Data_0_0.csv)
IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 3  # RGB画像の場合
BATCH_SIZE = 128  # 指定されたバッチサイズ
EPOCHS = 100  # 初期エポック数 (EarlyStoppingで調整されます)
LEARNING_RATE = 1e-4  # Adamの初期学習率 (転移学習で一般的な値)
TEST_SPLIT_RATIO = 0.1  # 全データに対するテストデータの割合
VALID_SPLIT_RATIO = 0.1  # 全データに対する検証データの割合 (残りの8割が訓練データ)

# CUDAデバイスの設定
print("利用可能なGPUデバイス:")
for i in range(cp.cuda.runtime.getDeviceCount()):
    print(f"GPU {i}: {cp.cuda.runtime.getDeviceProperties(i)['name']}")

# GPU 0を選択（RTX 6000 Ada Generation）
cp.cuda.Device(0).use()
print(f"\n選択されたGPU: {cp.cuda.runtime.getDeviceProperties(0)['name']}")


利用可能なGPUデバイス:
GPU 0: b'NVIDIA RTX 6000 Ada Generation'
GPU 1: b'NVIDIA T600'

選択されたGPU: b'NVIDIA RTX 6000 Ada Generation'


In [ ]:
# セル 3: CSVファイルから目的変数を抽出するヘルパー関数
def get_target_value(csv_filepath):
    """
    CSVファイルを読み込み、最初の30行を取得し、'T'列の平均値を計算します。
    """
    try:
        df = pd.read_csv(csv_filepath)
        # 最初の30行を対象とします
        target_df = df.head(30)

        if len(target_df) < 30:
            print(
                f"警告: {csv_filepath} の行数が30未満です ({len(target_df)}行)。利用可能な行で計算します。"
            )
            if len(target_df) == 0:
                return np.nan  # またはエラーとして処理

        # 'T'列の平均値を計算 (プロンプトのF列は6番目なので、CSVの'T'列と想定)
        target_mean = target_df["T"].mean()
        return target_mean
    except Exception as e:
        print(f"エラー: {csv_filepath} の処理中にエラーが発生しました: {e}")
        return np.nan


In [ ]:
# セル 4: データ読み込みと準備 (画像パスと目的変数)
image_files = sorted(
    glob.glob(os.path.join(IMG_BASE_PATH, "SingleI_*.bmp"))
)  # 画像ファイル名でソート
all_image_paths = []
all_targets = []

for img_path in image_files:
    basename = os.path.basename(img_path)  # 例: SingleI_0_0.bmp
    # ファイル名から 'X_Y' の部分を抽出 (例: '0_0')
    try:
        parts_str = basename.replace("SingleI_", "").replace(".bmp", "")
        parts = parts_str.split("_")

        if len(parts) == 2:
            csv_filename = f"Data_{parts[0]}_{parts[1]}.csv"
            csv_filepath = os.path.join(CSV_BASE_PATH, csv_filename)

            if os.path.exists(csv_filepath):
                target = get_target_value(csv_filepath)
                if not np.isnan(target):
                    all_image_paths.append(img_path)
                    all_targets.append(target)
                else:
                    print(
                        f"スキップ: {img_path} に対応する {csv_filepath} の目的変数が無効です。"
                    )
            else:
                print(
                    f"警告: {img_path} に対応するCSVファイルが見つかりません: {csv_filepath}"
                )
        else:
            print(f"警告: 画像ファイル名の形式が正しくありません: {img_path}")
    except Exception as e:
        print(f"エラー: 画像ファイル名 {img_path} の解析中にエラー: {e}")


all_image_paths = np.array(all_image_paths)
all_targets = np.array(all_targets)

print(f"\n{len(all_image_paths)} 個の画像と対応する目的変数を読み込みました。")
if len(all_image_paths) > 0:
    print(f"目的変数の例: {all_targets[:5]}")
else:
    raise ValueError(
        "画像と目的変数のペアが読み込めませんでした。パスやファイル名を確認してください。"
    )



31000 個の画像と対応する目的変数を読み込みました。
目的変数の例: [0.3666509  1.17850557 0.2556226  0.5822963  0.26573613]


In [ ]:
# セル 5: データ分割 (訓練データ、検証データ、テストデータ)
# まずテストデータを分離
train_val_paths, test_paths, train_val_targets, test_targets = train_test_split(
    all_image_paths,
    all_targets,
    test_size=TEST_SPLIT_RATIO,
    random_state=42,
    shuffle=True,
)

# 次に訓練データと検証データを分離 (残りのデータに対する割合で計算)
# (全データ中の検証データ割合) / (1 - 全データ中のテストデータ割合)
validation_split_from_train_val = VALID_SPLIT_RATIO / (1.0 - TEST_SPLIT_RATIO)
train_paths, val_paths, train_targets, val_targets = train_test_split(
    train_val_paths,
    train_val_targets,
    test_size=validation_split_from_train_val,
    random_state=42,
    shuffle=True,
)

print(f"訓練データ数: {len(train_paths)}")
print(f"検証データ数: {len(val_paths)}")
print(f"テストデータ数: {len(test_paths)}")


訓練データ数: 24800
検証データ数: 3100
テストデータ数: 3100


In [ ]:
# セル 6: VGG16用の画像読み込みと前処理関数
def load_and_preprocess_image(
    image_path_tensor,
):  # 引数名を変更してテンソルであることを明示
    # image_path_tensor は EagerTensor なので、Pythonの文字列に変換する
    # .numpy() でnumpy配列にし、.decode('utf-8')でバイト文字列を通常の文字列にデコード
    image_path_str = image_path_tensor.numpy().decode("utf-8")

    img = load_img(
        image_path_str,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        color_mode="rgb" if CHANNELS == 3 else "grayscale",
    )
    img_array = img_to_array(img)
    # VGG16のpreprocess_inputは、RGBをBGRに変換し、ImageNetの平均値でゼロセンタリングします。
    # グレースケールの場合、チャネルを3に複製するか、専用の前処理が必要です。ここではRGBを前提とします。
    img_array_preprocessed = tf.keras.applications.vgg16.preprocess_input(img_array)
    return img_array_preprocessed


In [ ]:
# セル 7: tf.data.Datasetの作成 (効率的なデータパイプライン)
def create_dataset(paths, targets, batch_size, shuffle=True, augment=False):
    # ファイルパスとラベルからデータセットを作成
    path_ds = tf.data.Dataset.from_tensor_slices(paths)

    # 画像の読み込みと前処理関数をマップ
    # AUTOTUNEはtf.dataに並列処理の数を動的に調整させます
    image_ds = path_ds.map(
        lambda x: tf.py_function(load_and_preprocess_image, [x], tf.float32),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    # py_functionの出力形状を設定
    def set_shape(img):
        img.set_shape((IMG_HEIGHT, IMG_WIDTH, CHANNELS))
        return img

    image_ds = image_ds.map(set_shape, num_parallel_calls=tf.data.AUTOTUNE)

    # ラベルのデータセットを作成
    label_ds = tf.data.Dataset.from_tensor_slices(tf.cast(targets, tf.float32))

    # 画像とラベルを結合
    ds = tf.data.Dataset.zip((image_ds, label_ds))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths))  # バッファサイズはデータ数程度

    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)  # 次のバッチを事前に読み込み
    return ds


train_dataset = create_dataset(train_paths, train_targets, BATCH_SIZE)
val_dataset = create_dataset(
    val_paths, val_targets, BATCH_SIZE, shuffle=False
)  # 検証データはシャッフルしない
test_dataset = create_dataset(
    test_paths, test_targets, BATCH_SIZE, shuffle=False
)  # テストデータはシャッフルしない


I0000 00:00:1747193422.781031 3110214 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 42541 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:18:00.0, compute capability: 8.9
I0000 00:00:1747193422.781937 3110214 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 2798 MB memory:  -> device: 1, name: NVIDIA T600, pci bus id: 0000:8a:00.0, compute capability: 7.5


In [ ]:
# セル 8: モデルの定義 (VGG16ベースの回帰モデル)
def build_vgg16_regressor(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), learning_rate=LEARNING_RATE
):
    # ImageNetで事前学習済みのVGG16モデルをロード (トップの分類層は含めない)
    # ★★★ VGG16インスタンスに 'vgg16_base_model' という名前を付ける ★★★
    base_model = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape,
        name="vgg16_base_model",
    )

    # VGG16ベースモデルの層を凍結 (転移学習の初期段階)
    for layer in base_model.layers:
        layer.trainable = False

    # 回帰のためのカスタム層を追加
    x = base_model.output
    x = GlobalAveragePooling2D()(x)  # 特徴量を画像ごとに1つのベクトルに変換
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)  # 正則化のためのドロップアウト
    # 回帰のための出力層: 1ニューロン、活性化関数は線形 (デフォルト)
    output_tensor = Dense(1, name="torque_output")(x)

    model = Model(inputs=base_model.input, outputs=output_tensor)

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="mean_squared_error",  # 回帰で一般的な損失関数
        metrics=["mean_absolute_error", tf.keras.metrics.RootMeanSquaredError()],
    )  # MAEは解釈しやすい

    return model


# モデルを再構築して、名前が適用されるようにする
model = build_vgg16_regressor()  # ★★★ 再度モデルをビルドする ★★★
model.summary()  # ここでvgg16_base_modelという名前のレイヤーがあるか確認


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ torque_output (Dense)           │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,977,857 (57.14 MB)

 Trainable params: 263,169 (1.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
# セル 9: 訓練時のコールバック設定
# EarlyStopping: 検証損失 (val_loss) が改善しなくなったら訓練を停止
early_stopping = EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
)

# ModelCheckpoint: 検証損失が改善したモデルのみを保存
model_checkpoint = ModelCheckpoint(
    "a.keras", monitor="val_loss", save_best_only=True, verbose=1
)

# ReduceLROnPlateau: 検証損失の改善が停滞した場合に学習率を動的に下げる
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=5, min_lr=1e-6, verbose=1
)

callbacks_list = [early_stopping, model_checkpoint, reduce_lr]


In [ ]:
# セル 10: モデルの訓練 (初期段階: VGG16ベースは凍結)
print("--- 初期訓練開始 (VGG16ベースは凍結) ---")
history = model.fit(
    train_dataset, epochs=EPOCHS, validation_data=val_dataset, callbacks=callbacks_list
)


--- 初期訓練開始 (VGG16ベースは凍結) ---
Epoch 1/100


2025-05-14 12:30:38.177317: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 2826 of 24800
2025-05-14 12:30:48.182142: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 5693 of 24800
2025-05-14 12:31:08.202544: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 11421 of 24800
2025-05-14 12:31:18.223826: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 14252 of 24800
2025-05-14 12:31:38.175308: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 19981 of 24800
2025-05-14 12:31:48.181669: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:5: Filling up shuffle buffer (this may 

  2/194 ━━━━━━━━━━━━━━━━━━━━ 19s 101ms/step - loss: 22.6004 - mean_absolute_error: 3.8175 - root_mean_squared_error: 4.7447   

I0000 00:00:1747193533.175151 3110889 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


102/194 ━━━━━━━━━━━━━━━━━━━━ 9s 104ms/step - loss: 11.7845 - mean_absolute_error: 2.7011 - root_mean_squared_error: 3.4128

KeyboardInterrupt: 

In [ ]:
# Cell 11: Plotting Training History
def plot_history(history, title_prefix=""):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Training Loss (MSE)")
    plt.plot(history.history["val_loss"], label="Validation Loss (MSE)")
    plt.title(f"{title_prefix}Model Loss")
    plt.ylabel("Loss (MSE)")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["mean_absolute_error"], label="Training MAE")
    plt.plot(history.history["val_mean_absolute_error"], label="Validation MAE")
    plt.title(f"{title_prefix}Model Mean Absolute Error (MAE)")
    plt.ylabel("MAE")
    plt.xlabel("Epoch")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# セル 12: 最良モデルのロード (EarlyStoppingでrestore_best_weights=Trueなら不要な場合もあるが念のため)
print("最良の重みをロードします...")
model.load_weights(
    "best_vgg16_regressor.keras"
)  # ModelCheckpointで保存された最良のモデル


In [ ]:
# セル 13: ファインチューニング (任意 - VGG16の一部レイヤーの凍結を解除して再訓練)
# VGG16の上位ブロック (例: block5, block4) の凍結を解除します。
# ファインチューニングは非常に低い学習率で行うべきです。

# 凍結解除するレイヤーの開始点を指定 (例: 'block4_conv1')
unfreeze_from_layer_name = "block4_conv1"

print(
    f"ファインチューニング: レイヤー '{unfreeze_from_layer_name}' 以降の凍結を解除します。"
)

# モデルのトップレベルのレイヤーを直接操作する
set_trainable = False
for layer in model.layers:
    # VGG16のレイヤーかどうかを名前で判定（'block'で始まるなど）
    # または、より具体的に操作したいレイヤー名を直接指定
    if layer.name == unfreeze_from_layer_name:
        set_trainable = True

    if set_trainable and layer.name.startswith(
        "block"
    ):  # VGG16のブロックレイヤーを対象とする
        layer.trainable = True
        # print(f"レイヤー {layer.name} を訓練可能にしました。")
    elif layer.name.startswith(
        "block"
    ):  # unfreeze_from_layer_name より前のブロックレイヤー
        layer.trainable = False
        # print(f"レイヤー {layer.name} を凍結しました。")
    # VGG16以外の追加したカスタムレイヤーは通常訓練可能のまま
    # (GlobalAveragePooling2D, Dense, Dropout など)
    # 必要であれば、これらの trainable 属性もここで制御可能

# ファインチューニング用にモデルを再コンパイル (より低い学習率で)
FINE_TUNE_LR = LEARNING_RATE / 10  # 例: 1e-5
optimizer_fine_tune = Adam(learning_rate=FINE_TUNE_LR)
model.compile(
    optimizer=optimizer_fine_tune,
    loss="mean_squared_error",
    metrics=["mean_absolute_error", tf.keras.metrics.RootMeanSquaredError()],
)

model.summary()  # 訓練可能なパラメータ数が増えていることを確認

print("\n--- ファインチューニング開始 ---")
fine_tune_epochs = 50  # ファインチューニングのエポック数 (状況に応じて調整)
# initial_epochをhistoryの最終エポックの次から開始するように設定
# history.epoch が空でないことを確認
initial_epoch_for_fine_tune = history.epoch[-1] + 1 if history.epoch else 0


# コールバックも更新 (ファインチューニング用に保存ファイル名変更など)
callbacks_fine_tune = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
        mode="min",
    ),
    ModelCheckpoint(
        "best_vgg16_regressor_finetuned.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
        mode="min",
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1, mode="min"
    ),  # ファインチューニングではより敏感に
]

history_fine_tune = model.fit(
    train_dataset,
    epochs=initial_epoch_for_fine_tune + fine_tune_epochs,  # 総エポック数
    initial_epoch=initial_epoch_for_fine_tune,  # 前の訓練の続きから開始
    validation_data=val_dataset,
    callbacks=callbacks_fine_tune,
)


In [ ]:
# セル 14: ファインチューニング後の訓練履歴のプロット
plot_history(history_fine_tune, "ファインチューニング後 ")


In [ ]:
# セル 15: テストセットでのモデル評価
from sklearn.metrics import r2_score  # R2スコア計算のためにインポート

print("\n--- テストセットでの評価 ---")
# (モデルのロード部分はそのまま)
# ...

test_loss, test_mae, test_rmse_metric_value = model.evaluate(
    test_dataset, verbose=1
)  # evaluateの返り値のRMSEに注意
print(f"テストデータの損失 (MSE): {test_loss:.4f}")
print(f"テストデータの平均絶対誤差 (MAE): {test_mae:.4f}")
print(
    f"テストデータのRMSE (Keras Metric): {test_rmse_metric_value:.4f}"
)  # KerasのメトリックとしてのRMSE

# 全テストデータに対する予測を一度に行う
all_test_predictions = []
all_test_true_values = []
for images, labels in test_dataset:  # バッチ処理されているためループで集める
    all_test_predictions.extend(model.predict_on_batch(images).flatten())
    all_test_true_values.extend(labels.numpy().flatten())

all_test_predictions = np.array(all_test_predictions)
all_test_true_values = np.array(all_test_true_values)

# R2スコアの計算
r2 = r2_score(all_test_true_values, all_test_predictions)
print(f"テストデータのR2スコア: {r2:.4f}")

# RMSEの再計算 (numpyベースで、Kerasメトリックと一致するか確認)
manual_rmse = np.sqrt(np.mean((all_test_predictions - all_test_true_values) ** 2))
print(f"テストデータのRMSE (numpy計算): {manual_rmse:.4f}")


In [ ]:
# セル 16: 予測の実行と結果の可視化 (任意)
# (all_test_true_values と all_test_predictions はセル15で計算済みとする)

plt.figure(figsize=(10, 8))  # 少し縦長に
plt.scatter(
    all_test_true_values, all_test_predictions, alpha=0.6, label="Predicted values"
)  # 翻訳
min_val = min(np.min(all_test_true_values), np.min(all_test_predictions))
max_val = max(np.max(all_test_true_values), np.max(all_test_predictions))
plt.plot(
    [min_val, max_val], [min_val, max_val], "r--", lw=2, label="Ideal prediction"
)  # 翻訳

# グラフにR2スコアとRMSEを表示
# (セル15で計算した r2 と manual_rmse を使用)
# R2とRMSEは国際的に通じるため、そのまま使用します
plt.text(
    0.05,
    0.95,
    f"R2: {r2:.3f}\nRMSE: {manual_rmse:.3f}",
    transform=plt.gca().transAxes,  # 軸相対座標
    fontsize=14,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.5", fc="yellow", alpha=0.5),
)


plt.xlabel("Actual Torque Value (FEM [Nm])")  # 翻訳 (元論文の表記に合わせる)
plt.ylabel("Predicted Torque Value (CNN [Nm])")  # 翻訳 (元論文の表記に合わせる)
plt.title("Actual Torque Value vs. Predicted Torque Value on Test Data")  # 翻訳
plt.legend(loc="lower right")
plt.grid(True)
# 軸の範囲を0から開始し、最大値はデータの最大値より少し大きめに設定
plt.xlim(left=0, right=max_val * 1.05)
plt.ylim(bottom=0, top=max_val * 1.05)
plt.gca().set_aspect("equal", adjustable="box")  # アスペクト比を1:1に
plt.show()


## モデルの精度のみ実行したい．
cellの１から7までを実行し以下を実行してください．

In [ ]:
# Assume Cells 1-7 have been executed and variables like
# test_paths, test_targets, test_dataset, and the function build_vgg16_regressor are defined.


# --- Step 8: Load the trained model ---
# Make sure to define IMG_HEIGHT, IMG_WIDTH, CHANNELS, LEARNING_RATE if not already defined in current session
# (These should be defined in Cell 2 of your original notebook)
def build_vgg16_regressor(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), learning_rate=LEARNING_RATE
):
    # ImageNetで事前学習済みのVGG16モデルをロード (トップの分類層は含めない)
    # ★★★ VGG16インスタンスに 'vgg16_base_model' という名前を付ける ★★★
    base_model = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape,
        name="vgg16_base_model",
    )

    # VGG16ベースモデルの層を凍結 (転移学習の初期段階)
    for layer in base_model.layers:
        layer.trainable = False

    # 回帰のためのカスタム層を追加
    x = base_model.output
    x = GlobalAveragePooling2D()(x)  # 特徴量を画像ごとに1つのベクトルに変換
    x = Dense(512, activation="relu")(x)
    x = Dropout(0.5)(x)  # 正則化のためのドロップアウト
    # 回帰のための出力層: 1ニューロン、活性化関数は線形 (デフォルト)
    output_tensor = Dense(1, name="torque_output")(x)

    model = Model(inputs=base_model.input, outputs=output_tensor)

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="mean_squared_error",  # 回帰で一般的な損失関数
        metrics=["mean_absolute_error", tf.keras.metrics.RootMeanSquaredError()],
    )  # MAEは解釈しやすい

    return model


# Re-build the model structure
model = build_vgg16_regressor(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), learning_rate=LEARNING_RATE
)  # Or the learning rate used for fine-tuning if that was best

# Load the weights of your best trained model
# Replace 'your_best_model_weights.keras' with the actual filename
# e.g., 'best_vgg16_regressor_finetuned.keras' or 'best_vgg16_regressor.keras'
best_model_path = (
    "best_vgg16_regressor_finetuned.keras"  # OR 'best_vgg16_regressor.keras'
)
try:
    model.load_weights(best_model_path)
    print(f"Successfully loaded weights from {best_model_path}")
except Exception as e:
    print(f"Error loading weights from {best_model_path}: {e}")
    print(
        "Please ensure the model path is correct and the model architecture matches the saved weights."
    )
    # If loading fails, you might need to re-run the training cells or check the path.

import numpy as np  # Ensure numpy is imported

# --- Step 9: Re-calculate predictions and metrics on the test set ---
from sklearn.metrics import r2_score  # For R2 score calculation

print("\n--- Evaluating on Test Set (re-calculation) ---")

# Evaluate to get Keras metrics (optional, but good for consistency check)
test_loss, test_mae, test_rmse_keras_metric = model.evaluate(test_dataset, verbose=1)
print(f"Test Loss (MSE) from Keras evaluate: {test_loss:.4f}")
print(f"Test Mean Absolute Error (MAE) from Keras evaluate: {test_mae:.4f}")
print(f"Test RMSE from Keras evaluate: {test_rmse_keras_metric:.4f}")

# Get all predictions for the test set
all_test_predictions_list = []
all_test_true_values_list = []

# Iterate through the test_dataset to get all predictions and true values
# test_dataset should be created as in Cell 7
for images_batch, labels_batch in test_dataset:
    predictions_batch = model.predict_on_batch(images_batch)
    all_test_predictions_list.extend(predictions_batch.flatten())
    all_test_true_values_list.extend(labels_batch.numpy().flatten())

all_test_predictions = np.array(all_test_predictions_list)
all_test_true_values = np.array(all_test_true_values_list)

# Calculate R2 score
r2 = r2_score(all_test_true_values, all_test_predictions)
print(f"Test R2 Score: {r2:.4f}")

# Calculate RMSE manually (as in your original Cell 15)
manual_rmse = np.sqrt(np.mean((all_test_predictions - all_test_true_values) ** 2))
print(f"Test RMSE (manual calculation): {manual_rmse:.4f}")


# --- Step 10: Plotting the results (English version of Cell 16) ---
import matplotlib.pyplot as plt  # Ensure matplotlib.pyplot is imported

plt.figure(figsize=(10, 8))  # Slightly taller figure
plt.scatter(
    all_test_true_values, all_test_predictions, alpha=0.6, label="Predicted Values"
)

# Determine plot limits
min_val_plot = min(np.min(all_test_true_values), np.min(all_test_predictions))
max_val_plot = max(np.max(all_test_true_values), np.max(all_test_predictions))

# Ideal prediction line (y=x)
plt.plot(
    [min_val_plot, max_val_plot],
    [min_val_plot, max_val_plot],
    "r--",
    lw=2,
    label="Ideal Prediction",
)

# Add R2 score and RMSE to the plot
# Using the 'r2' and 'manual_rmse' calculated above
plt.text(
    0.05,
    0.95,
    f"R2: {r2:.3f}\nRMSE: {manual_rmse:.3f}",
    transform=plt.gca().transAxes,  # Axes relative coordinates
    fontsize=14,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.5", fc="lightgrey", alpha=0.8),
)  # Changed background color slightly

plt.xlabel("Actual Torque (FEM [Nm])")  # As per the original paper's notation
plt.ylabel("Predicted Torque (CNN [Nm])")  # As per the original paper's notation
plt.title("Actual vs. Predicted Torque on Test Set")
plt.legend(loc="lower right")
plt.grid(True)

# Set axis limits starting from 0 and slightly larger than max data value
# Adjust xlim and ylim if your data can be negative or has a different typical range
axis_min_limit = 0  # Assuming torque is non-negative
axis_max_limit = max_val_plot * 1.05
plt.xlim(left=axis_min_limit, right=axis_max_limit)
plt.ylim(bottom=axis_min_limit, top=axis_max_limit)

plt.gca().set_aspect("equal", adjustable="box")  # Make aspect ratio 1:1
plt.show()


## 訓練データセットから画像データを取得し，予測を行う

In [ ]:
# (必要なインポートや train_dataset が定義されている前提)
import time  # 時間計測用 (任意)

import numpy as np
import tensorflow as tf

print("\n--- Collecting processed training images into variable X ---")

# 訓練データセットから前処理済みの画像データを収集するためのリスト
all_train_images_list = []
all_train_labels_list = []
print("Gathering processed images from train_dataset...")
start_time = time.time()

# train_dataset をイテレートして画像データを抽出
# train_dataset が定義されていない場合は、セル7などを再実行して作成してください。
# (train_dataset がシャッフルされている場合、元の train_paths との順序は保証されません)
try:
    num_batches = tf.data.experimental.cardinality(train_dataset).numpy()
    if (
        num_batches == tf.data.experimental.UNKNOWN_CARDINALITY
        or num_batches == tf.data.experimental.INFINITE_CARDINALITY
    ):
        print("Warning: Could not determine the exact number of batches.")
        num_batches = -1
except:  # tf.data.experimental.cardinality が使えない/失敗する場合
    print(
        "Warning: Could not determine the number of batches using tf.data.experimental.cardinality."
    )
    num_batches = -1

processed_samples = 0
batch_counter = 0
for (
    images_batch,
    images_labels,
) in train_dataset:  # ラベル情報は今回は使用しないので _ で受け取る
    # バッチ内の画像データをNumPy配列に変換してリストに追加
    all_train_images_list.append(images_batch.numpy())
    all_train_labels_list.append(images_labels.numpy())
    processed_samples += images_batch.shape[0]
    batch_counter += 1
    if num_batches > 0:
        print(
            f"Processing batch {batch_counter}/{num_batches}, Samples processed: {processed_samples}",
            end="\r",
        )
    else:
        print(
            f"Processing batch {batch_counter}, Samples processed: {processed_samples}",
            end="\r",
        )


print("\nConcatenating collected image batches...")
# リストに格納された複数のバッチ（NumPy配列）を一つの大きなNumPy配列に結合
if all_train_images_list:
    X_train_images = np.concatenate(all_train_images_list, axis=0)
    # ユーザーが指定した変数名 'X' に代入
    X = X_train_images
    end_time = time.time()
    print(f"\nFinished collecting images in {end_time - start_time:.2f} seconds.")
    print(
        f"Processed training images are now stored in variable 'X' with shape: {X.shape}"
    )
    # メモリ使用量の概算を表示 (GiB単位)
    print(f"Approximate memory usage of X: {X.nbytes / (1024**3):.2f} GiB")

    # 必要であれば、元のリストを削除してメモリを解放
    del all_train_images_list

else:
    print("\nNo images were collected from train_dataset. Variable 'X' is not created.")
    X = None  # XをNoneに設定

# これで変数 X に前処理済みの訓練画像データ (NumPy配列) が格納されました。
# この後の処理で、この変数 X を使用できますが、メモリ使用量に注意してください。


# model.predict() を使って予測を実行
try:
    # batch_size を指定すると、メモリ使用量を抑えられる
    # ご自身の環境のメモリに合わせて調整してください (例: 64, 32, 16)
    # 指定しない場合は、TensorFlowが自動で決定しようとします
    predict_batch_size = 64  # バッチサイズを調整
    print(f"Starting prediction with batch_size={predict_batch_size}...")
    Y_predictions_raw = model.predict(X, batch_size=predict_batch_size, verbose=1)

    # 予測結果は通常 (サンプル数, 1) の形状なので、(サンプル数,) に平坦化する
    Y = Y_predictions_raw.flatten()

    end_time = time.time()
    print(f"\nFinished prediction in {end_time - start_time:.2f} seconds.")
    print(f"Predictions are now stored in variable 'Y' with shape: {Y.shape}")

    # 予測結果の最初のいくつかを表示 (確認用)
    print(f"First 5 predictions in Y: {Y[:5]}")

except Exception as e:
    print(f"\nAn error occurred during prediction: {e}")
    print("Consider reducing the 'predict_batch_size' if this is a memory issue.")
    Y = None  # エラーが発生した場合は Y を None に設定

# これで変数 Y に、入力 X (訓練画像データ) に対する予測値が格納されました。

X = X.reshape(X.shape[0], X.shape[1] * X.shape[2] * X.shape[3])
print(X.shape, Y.shape, all_train_labels_list)


In [ ]:
Y_true_train_data = all_train_labels_list
print(Y_true_train_data.shape)


## X, Y, 正解値をファイルに書き込む

In [ ]:
import os
import time

import numpy as np

# --- Save X, Y (predictions), and Y_true (actual labels) to a compressed .npz file ---

# 保存するファイル名を定義 (拡張子は .npz)
filename = "training_data_preprocessed_predicted_and_true_labels.npz"  # ファイル名を少し変更して内容を反映
save_directory = "./data/20250425_data"  # 保存先のディレクトリ
os.makedirs(save_directory, exist_ok=True)  # 保存先ディレクトリがなければ作成
full_filepath = os.path.join(save_directory, filename)

print(f"\n--- Saving data to {full_filepath} ---")

# ★★★ 元の訓練データの正解ラベルを格納している変数名を指定 ★★★
# 例: Y_true_train, train_targets, all_targets_train など、ご自身のコードに合わせてください。
# ここでは 'Y_true_train_data' という名前の変数に正解ラベルが入っていると仮定します。
# この変数は、X や Y と同じサンプル数・同じ順序である必要があります。

# ダミーの Y_true_train_data (実際のデータに置き換えてください)
if "X" in locals() and X is not None:  # Xの形状に基づいてダミーを作成
    if "Y_true_train_data" not in locals() or Y_true_train_data is None:
        print(
            "Warning: 'Y_true_train_data' not found. Using dummy true labels for saving demonstration."
        )
        Y_true_train_data = np.random.rand(X.shape[0]).astype(
            np.float32
        )  # Xと同じサンプル数のダミー正解ラベル
else:
    if "Y_true_train_data" not in locals() or Y_true_train_data is None:
        print(
            "Warning: 'X' not found. Cannot create dummy Y_true_train_data of appropriate size."
        )
        Y_true_train_data = None


# 変数 X (画像データ), Y (予測値), Y_true_train_data (正解ラベル) が存在するか確認
if (
    "X" in locals()
    and "Y" in locals()
    and "Y_true_train_data" in locals()
    and X is not None
    and Y is not None
    and Y_true_train_data is not None
):

    # Y と Y_true_train_data の形状が一致するか確認 (Yは予測値なのでXのサンプル数と同じはず)
    if X.shape[0] != Y.shape[0] or X.shape[0] != Y_true_train_data.shape[0]:
        print(
            "Error: Mismatch in the number of samples between X, Y, and Y_true_train_data."
        )
        print(
            f"X shape[0]: {X.shape[0]}, Y shape[0]: {Y.shape[0]}, Y_true_train_data shape[0]: {Y_true_train_data.shape[0]}"
        )
    else:
        start_time = time.time()
        try:
            # np.savez_compressed を使用して圧縮形式で保存
            # キーワード引数で各配列に名前を付ける
            np.savez_compressed(
                full_filepath,
                x_data=X,  # 前処理済み画像データ
                y_predicted_data=Y,  # モデルによる予測値
                y_true_actual_data=Y_true_train_data,  # ★★★ 元の訓練データの正解ラベル ★★★
            )

            end_time = time.time()
            print(f"Successfully saved data to {full_filepath}")
            print(f"  X shape: {X.shape}")
            print(f"  Y (predicted) shape: {Y.shape}")
            print(f"  Y_true (actual) shape: {Y_true_train_data.shape}")
            print(f"Saving took {end_time - start_time:.2f} seconds.")

            file_size_mb = os.path.getsize(full_filepath) / (1024**2)
            print(f"File size: {file_size_mb:.2f} MB")

        except Exception as e:
            print(f"An error occurred while saving the file: {e}")
else:
    missing_vars = []
    if "X" not in locals() or X is None:
        missing_vars.append("'X'")
    if "Y" not in locals() or Y is None:
        missing_vars.append("'Y' (predictions)")
    if "Y_true_train_data" not in locals() or Y_true_train_data is None:
        missing_vars.append("'Y_true_train_data' (true labels)")
    print(
        f"Error: Variables {', '.join(missing_vars)} not found or are None. Cannot save."
    )

# メモリを解放したい場合は、ここで X や Y, Y_true_train_data を削除することも検討
# del X
# del Y
# del Y_true_train_data
# print("Variables X, Y, and Y_true_train_data deleted from memory.")


In [ ]:
# import os
# import time

# import numpy as np

# # --- Save X and Y to a compressed .npz file ---

# # 保存するファイル名を定義 (拡張子は .npz)
# filename = "training_data_preprocessed_and_predicted.npz"
# save_directory = (
#     "./data/20250425_data"  # 保存先のディレクトリ (カレントディレクトリに保存)
# )
# full_filepath = os.path.join(save_directory, filename)

# print(f"\n--- Saving X and Y to {full_filepath} ---")

# # 変数 X と Y が存在するか確認
# if "X" in locals() and "Y" in locals() and X is not None and Y is not None:
#     start_time = time.time()
#     try:
#         # np.savez_compressed を使用して圧縮形式で保存
#         # キーワード引数で配列に名前を付ける (例: x_data=X, y_data=Y)
#         # これにより、読み込み時に名前でアクセスできる
#         np.savez_compressed(full_filepath, x_data=X, y_data=Y)

#         end_time = time.time()
#         print(
#             f"Successfully saved X (shape: {X.shape}) and Y (shape: {Y.shape}) to {full_filepath}"
#         )
#         print(f"Saving took {end_time - start_time:.2f} seconds.")
#         # ファイルサイズの確認 (任意)
#         file_size_mb = os.path.getsize(full_filepath) / (1024**2)
#         print(f"File size: {file_size_mb:.2f} MB")

#     except Exception as e:
#         print(f"An error occurred while saving the file: {e}")

# else:
#     print("Error: Variables 'X' or 'Y' not found or are None. Cannot save.")

# # メモリを解放したい場合は、ここで X や Y を削除することも検討
# # del X
# # del Y
# # print("Variables X and Y deleted from memory.")


## 画像データX, 予測値Y, 正解値の読み込み

In [ ]:
import os

import numpy as np

# 保存したファイル名を指定
filename = "training_data_preprocessed_predicted_and_true_labels.npz"
load_directory = "./data/20250425_data"  # 保存したディレクトリと同じ場所を指定
full_filepath = os.path.join(load_directory, filename)

# 読み込み時のイメージ
loaded_data = np.load(full_filepath)
X_loaded = loaded_data["x_data"]
Y_predicted_loaded = loaded_data["y_predicted_data"]
Y_true_loaded = loaded_data["y_true_actual_data"]


In [ ]:
print(X_loaded.shape, Y_predicted_loaded.shape, Y_true_loaded.shape)
Y_true_loaded


(24800, 150528) (24800,) (24800,)


array([0.45223007, 1.513916  , 1.2277174 , ..., 0.02406127, 1.7692201 ,
       0.9797895 ], dtype=float32)

In [ ]:
# print("\nComparing elements using np.array_equal()...")
# are_equal = np.array_equal(X, X_loaded)

# if are_equal:
#     print("\nResult: SUCCESS! X and X_loaded have the same shape and elements.")
# else:
#     print(
#         "\nResult: FAILURE! X and X_loaded do NOT have the same elements (even though shapes match)."
#     )
#
# Comparing elements using np.array_equal()...

# Result: SUCCESS! X and X_loaded have the same shape and elements.


## AIMEの実装

In [ ]:
import cupy as cp
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from aime_xai.core_gpu import AIME_GPU


In [ ]:
aime_gpu = AIME_GPU()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_loaded)


In [ ]:
print(f"Shape of X_scaled: {X_scaled.shape}")
print(f"Data type of X_scaled: {X_scaled.dtype}")
print(f"Estimated memory size of X_scaled (CPU): {X_scaled.nbytes / (1024**3):.2f} GB")


In [ ]:
X_cp = cp.asarray(X_scaled)
Y_cp = cp.asarray(Y_predicted_loaded)
if Y_cp.ndim == 1:
    print(f"Reshaping Y_cp to ({Y_cp.shape[0]}, 1)")
    Y_cp = Y_cp.reshape(-1, 1)


In [ ]:
print(X_cp.shape, Y_cp.shape)


In [ ]:
aime_gpu.create_explainer(X_cp, Y_cp, normalize=False)


In [ ]:
aime_gpu.A_dagger.shape


In [ ]:
feature_names = [f"pixel_{i}" for i in range(X_scaled.shape[1])]


In [ ]:
# グローバル特徴重要度の計算と可視化
df_global_importance = aime_gpu.global_feature_importance(
    feature_names=feature_names, top_k=10
)


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
import numpy as np
import pandas as pd  # df_globalがPandas DataFrameの場合に備えて

# --- 前提となる変数 (これらの変数が適切に設定されている必要があります) ---
# aime_gpu: AIME_GPU のインスタンスで、既に explainer が作成されている
# IMAGE_SIZE: 元の画像のサイズ (タプル, 例: (224, 224))
# NUM_CHANNELS: 画像のチャンネル数 (例: 3 for RGB)

try:
    # --- IMAGE_SIZE と NUM_CHANNELS の定義確認 ---
    if "IMAGE_SIZE" not in globals():
        IMAGE_SIZE = (224, 224)  # フォールバック例
        print(f"Warning: IMAGE_SIZE was not defined. Using default: {IMAGE_SIZE}")
    if "NUM_CHANNELS" not in globals():
        NUM_CHANNELS = 3  # デフォルトをRGBと仮定
        print(f"Warning: NUM_CHANNELS was not defined. Assuming {NUM_CHANNELS}.")

    expected_num_features = IMAGE_SIZE[0] * IMAGE_SIZE[1] * NUM_CHANNELS

    # --- feature_names の準備 (AIME APIが要求する場合) ---
    # DataFrameの列名が 'pixel_0', 'pixel_1', ... のようになっているので、
    # それに合わせたリストを作成する
    # AIMEのAPIがこれを必要としない場合は、この部分は不要かもしれません。
    feature_names = [f"pixel_{i}" for i in range(expected_num_features)]

    # --- AIMEからグローバル特徴量重要度を取得 ---
    # class_names は回帰なのでNoneにするか、APIが許せば省略
    df_global_aime = aime_gpu.global_feature_importance_without_viz(
        feature_names=feature_names,  # APIがfeature_namesを必要とする場合
        class_names=None,  # 回帰タスクなのでNoneまたは省略を試す
    )

    # df_global_aime が cuDF DataFrame の場合は Pandas DataFrame に変換
    if hasattr(df_global_aime, "to_pandas"):
        df_global_aime = df_global_aime.to_pandas()

    # DataFrameの形式を確認し、値を取得
    if not isinstance(df_global_aime, pd.DataFrame):
        raise TypeError(f"AIME did not return a DataFrame. Got: {type(df_global_aime)}")
    if df_global_aime.shape[0] != 1 or df_global_aime.shape[1] != expected_num_features:
        raise ValueError(
            f"AIME DataFrame shape mismatch. Expected (1, {expected_num_features}), got {df_global_aime.shape}"
        )

    # "Mean" 行の値を取得 (インデックス名が 'Mean' であることを前提)
    if "Mean" in df_global_aime.index:
        global_importance_values = df_global_aime.loc["Mean"].values
    else:  # インデックス名が不明な場合は最初の行を取得
        print("Warning: Index 'Mean' not found in AIME DataFrame. Using the first row.")
        global_importance_values = df_global_aime.iloc[0].values

    # 結果がCuPy配列ならNumPy配列に変換 (iloc[0].valuesでNumPyになることが多いが念のため)
    if isinstance(global_importance_values, cp.ndarray):
        global_importance_values = cp.asnumpy(global_importance_values)

    # 1次元配列になっていることを確認
    if global_importance_values.ndim != 1:
        global_importance_values = (
            global_importance_values.flatten()
        )  # 万が一多次元ならフラット化
    if len(global_importance_values) != expected_num_features:
        raise ValueError(
            f"Global importance values have unexpected length: {len(global_importance_values)}. "
            f"Expected {expected_num_features}."
        )

    # --- 重要度を画像の形状にリシェイプ ---
    importance_reshaped = global_importance_values.reshape(
        IMAGE_SIZE[0], IMAGE_SIZE[1], NUM_CHANNELS
    )

    # --- ヒートマップの表示 ---

    # 正規化のための全体的な最小値・最大値
    # (重要度には負の値も含まれる可能性があるため、単純な0-1正規化とは異なるアプローチも考慮)
    overall_min_imp = importance_reshaped.min()
    overall_max_imp = importance_reshaped.max()
    print(
        f"Overall importance range: min={overall_min_imp:.4f}, max={overall_max_imp:.4f}"
    )

    # オプション1: 各チャンネルを個別に表示 (共通のカラースケールを使用)
    fig1, axes1 = plt.subplots(
        1, NUM_CHANNELS, figsize=(6 * NUM_CHANNELS, 5.5)
    )  # サイズ調整
    if NUM_CHANNELS == 1:
        axes1 = [axes1]  # axes1がリストになるように

    channel_names = ["Red Channel", "Green Channel", "Blue Channel"]  # RGBの場合
    if NUM_CHANNELS == 1:
        channel_names = ["Grayscale Channel"]

    # 共通のカラースケールで表示（vmin, vmaxを指定）
    # 発散的カラーマップ 'coolwarm' や 'RdBu_r' などが負の値を含む場合に適している
    # 全て正の値なら 'inferno', 'viridis', 'hot' など
    # データ範囲に応じてcmapとvmin/vmaxを選択
    cmap_to_use = "coolwarm" if overall_min_imp < 0 else "inferno"

    for i in range(NUM_CHANNELS):
        channel_importance = importance_reshaped[:, :, i]
        ax = axes1[i]
        # カラーバーを各サブプロットに追加する場合
        im = ax.imshow(
            channel_importance,
            cmap=cmap_to_use,
            vmin=overall_min_imp,
            vmax=overall_max_imp,
        )
        ax.set_title(
            f"{channel_names[i] if i < len(channel_names) else f'Channel {i}'}"
        )
        ax.axis("off")
        fig1.colorbar(im, ax=ax, shrink=0.7, label="Feature Importance")  # shrink調整

    fig1.suptitle("Global Feature Importance (Per Channel)", fontsize=16)
    fig1.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # オプション2: 全チャンネルの重要度の絶対値の平均を1枚のヒートマップとして表示
    # (負の重要度も「影響度」として同じように扱いたい場合)
    importance_aggregated = np.mean(np.abs(importance_reshaped), axis=2)
    agg_min = importance_aggregated.min()
    agg_max = importance_aggregated.max()

    fig2, ax2 = plt.subplots(1, 1, figsize=(7, 6))
    # 絶対値の平均なので、通常は 'inferno' や 'viridis' などのシーケンシャルなカラーマップ
    im_agg = ax2.imshow(
        importance_aggregated, cmap="inferno", vmin=agg_min, vmax=agg_max
    )
    ax2.set_title("Global Feature Importance (Mean Absolute Value over Channels)")
    ax2.axis("off")
    fig2.colorbar(im_agg, ax=ax2, shrink=0.8, label="Mean Absolute Importance")
    plt.show()

except Exception as e:
    print(f"An error occurred during heatmap generation: {e}")
    import traceback

    traceback.print_exc()


## 局所的特徴重要度

In [ ]:
# index_to_analyze = 900  # ここで分析したい画像のインデックスを指定
# x_sample = X_loaded[index_to_analyze]
# y_sample = Y_predicted_loaded[index_to_analyze]
# y_true_sample = Y_true_loaded[index_to_analyze]
# x_sample_cp = cp.asarray(x_sample)
# y_sample_cp = cp.array([y_sample])

# # ローカル特徴重要度の計算
# df_local_importance = aime_gpu.local_feature_importance_without_viz(
#     x_sample_cp,
#     y_sample_cp,
#     feature_names=feature_names,
#     scale=True,
#     scaler=scaler,
#     ignore_zero_features=True,
# )

# import cupy as cp  # df_local_importance が CuPy 配列の場合に備えて
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd  # df_local_importance が DataFrame の場合

# # ----- STEP 0: 必要な変数の定義 (実際の値に置き換えてください) -----
# # これらは通常、ノートブックの前のセルで定義・計算・ロードされているはずです。

# # 例: 画像の寸法 (VGG16の場合など)
# IMG_HEIGHT = 224
# IMG_WIDTH = 224
# CHANNELS = 3  # RGB画像の場合。グレースケールなら1

# # ----- STEP 1: 分析対象のデータ準備 -----

# # --- 画像データの取得とリシェイプ ---
# if not all(
#     var in locals() or var in globals()
#     for var in ["IMG_HEIGHT", "IMG_WIDTH", "CHANNELS"]
# ):
#     print("Error: IMG_HEIGHT, IMG_WIDTH, or CHANNELS not defined.")
#     raise NameError(
#         "Image dimensions (IMG_HEIGHT, IMG_WIDTH, CHANNELS) are not defined."
#     )

# if (
#     "X_loaded" not in locals()
#     or X_loaded is None
#     or not isinstance(X_loaded, np.ndarray)
# ):
#     print("Error: 'X_loaded' not found, is None, or is not a NumPy array.")
#     raise NameError("'X_loaded' is not properly defined or loaded.")

# if not (0 <= index_to_analyze < X_loaded.shape[0]):
#     print(
#         f"Error: index_to_analyze ({index_to_analyze}) is out of range for X_loaded (0 to {X_loaded.shape[0]-1})."
#     )
#     raise IndexError("index_to_analyze is out of bounds.")

# x_sample_flat_preprocessed = X_loaded[index_to_analyze]
# try:
#     target_size = IMG_HEIGHT * IMG_WIDTH * CHANNELS
#     if (
#         CHANNELS == 1 and x_sample_flat_preprocessed.size == IMG_HEIGHT * IMG_WIDTH
#     ):  # グレースケールで平坦化済みの場合
#         target_size = IMG_HEIGHT * IMG_WIDTH
#         x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
#             IMG_HEIGHT, IMG_WIDTH, 1
#         )  # チャンネル次元を保持
#     elif x_sample_flat_preprocessed.size == target_size:
#         x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
#             IMG_HEIGHT, IMG_WIDTH, CHANNELS
#         )
#     else:
#         raise ValueError(
#             f"Flat image size {x_sample_flat_preprocessed.size} does not match dimensions "
#             f"({IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS} or {IMG_HEIGHT}x{IMG_WIDTH} for grayscale)."
#         )
#     print(
#         f"Original preprocessed x_sample (index {index_to_analyze}) reshaped to: {x_sample_reshaped_preprocessed.shape}"
#     )
# except ValueError as ve:
#     print(f"Error reshaping x_sample: {ve}")
#     raise

# # --- 特徴重要度の取得とリシェイプ ---
# if "df_local_importance" not in locals() or df_local_importance is None:
#     print("Error: 'df_local_importance' not found or is None.")
#     print(
#         "Please ensure local feature importance has been calculated for the selected sample."
#     )
#     raise NameError("'df_local_importance' is not defined.")

# if isinstance(df_local_importance, pd.DataFrame):
#     if (
#         df_local_importance.shape[0] == 1 and df_local_importance.shape[1] > 0
#     ):  # 単一サンプルのDataFrameを想定
#         local_importance_values_flat = df_local_importance.iloc[0].to_numpy()
#     elif (
#         df_local_importance.shape[0] > 1
#         and df_local_importance.shape[0] > index_to_analyze
#     ):  # 複数行ある場合
#         print(
#             f"Warning: df_local_importance has multiple rows. Selecting row at index {index_to_analyze}."
#         )
#         local_importance_values_flat = df_local_importance.iloc[
#             index_to_analyze
#         ].to_numpy()
#     else:
#         raise ValueError(
#             f"df_local_importance shape {df_local_importance.shape} is not suitable or index is out of bounds."
#         )
# elif isinstance(df_local_importance, cp.ndarray):
#     # CuPy配列の場合、1サンプル分になっているか確認が必要
#     if df_local_importance.ndim == 1:
#         local_importance_values_flat = cp.asnumpy(df_local_importance)
#     elif df_local_importance.ndim == 2 and df_local_importance.shape[0] == 1:
#         local_importance_values_flat = cp.asnumpy(df_local_importance.flatten())
#     elif (
#         df_local_importance.ndim == 2
#         and df_local_importance.shape[0] > index_to_analyze
#     ):
#         print(
#             f"Warning: CuPy df_local_importance has multiple rows. Selecting row at index {index_to_analyze}."
#         )
#         local_importance_values_flat = cp.asnumpy(
#             df_local_importance[index_to_analyze].flatten()
#         )
#     else:
#         raise ValueError(
#             f"CuPy df_local_importance shape {df_local_importance.shape} is not suitable."
#         )
# elif isinstance(df_local_importance, np.ndarray):
#     if df_local_importance.ndim == 1:
#         local_importance_values_flat = df_local_importance
#     elif df_local_importance.ndim == 2 and df_local_importance.shape[0] == 1:
#         local_importance_values_flat = df_local_importance.flatten()
#     elif (
#         df_local_importance.ndim == 2
#         and df_local_importance.shape[0] > index_to_analyze
#     ):
#         print(
#             f"Warning: NumPy df_local_importance has multiple rows. Selecting row at index {index_to_analyze}."
#         )
#         local_importance_values_flat = df_local_importance[index_to_analyze].flatten()
#     else:
#         raise ValueError(
#             f"NumPy df_local_importance shape {df_local_importance.shape} is not suitable."
#         )
# else:
#     raise TypeError("df_local_importance is not of a recognized type.")

# try:
#     # 特徴重要度がピクセル単位 (H*W) か、チャネル含む (H*W*C) かで処理を分ける
#     if local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH:
#         importance_heatmap_2d = local_importance_values_flat.reshape(
#             IMG_HEIGHT, IMG_WIDTH
#         )
#     elif local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH * CHANNELS:
#         temp_heatmap_3d = local_importance_values_flat.reshape(
#             IMG_HEIGHT, IMG_WIDTH, CHANNELS
#         )
#         importance_heatmap_2d = np.mean(temp_heatmap_3d, axis=2)  # チャネル平均
#         print("Importance values reshaped from (H,W,C) by taking mean over channels.")
#     else:
#         raise ValueError(
#             f"Size of local_importance_values_flat ({local_importance_values_flat.size}) "
#             f"does not match expected image dimensions (H*W or H*W*C)."
#         )
#     print(f"Reshaped importance heatmap to: {importance_heatmap_2d.shape}")
# except ValueError as ve:
#     print(f"Error reshaping importance_heatmap_2d: {ve}")
#     raise

# # ----- STEP 1.5: 特徴重要度の正規化 -----
# all_importance_values_for_norm = (
#     importance_heatmap_2d.flatten()
# )  # 2Dヒートマップから正規化範囲を計算

# local_min = all_importance_values_for_norm.min()
# local_max = all_importance_values_for_norm.max()
# print(
#     f"Calculated local_min: {local_min:.4f}, local_max: {local_max:.4f} from the reshaped importance heatmap."
# )

# if (local_max - local_min) > 1e-6:
#     importance_heatmap_normalized = (importance_heatmap_2d - local_min) / (
#         local_max - local_min
#     )
# else:
#     importance_heatmap_normalized = np.zeros_like(importance_heatmap_2d)
#     print(
#         "Warning: All importance values in the heatmap are nearly identical. Normalized heatmap will be all zeros."
#     )
# importance_heatmap_normalized = np.clip(importance_heatmap_normalized, 0, 1)
# print(f"Shape of normalized importance heatmap: {importance_heatmap_normalized.shape}")


# # ----- STEP 2: 表示用関数の定義 -----
# def display_denormalized_vgg16_for_subplot(ax, img_data_reshaped_preprocessed, title):
#     if not (
#         img_data_reshaped_preprocessed.ndim == 3
#         and img_data_reshaped_preprocessed.shape[-1] == 3
#     ):
#         ax.text(
#             0.5,
#             0.5,
#             f"Cannot display original image:\nExpected 3D RGB image,\ngot shape {img_data_reshaped_preprocessed.shape}",
#             ha="center",
#             va="center",
#             fontsize=9,
#         )
#         ax.set_title(title, fontsize=10)
#         ax.axis("off")
#         return

#     img_restored = img_data_reshaped_preprocessed.copy()
#     # VGG16の逆前処理: 平均値を足し、BGRをRGBに変換
#     img_restored[..., 0] += 103.939  # Blue
#     img_restored[..., 1] += 116.779  # Green
#     img_restored[..., 2] += 123.68  # Red
#     img_restored = img_restored[..., ::-1]  # BGR to RGB
#     img_restored = np.clip(img_restored, 0, 255).astype(
#         np.uint8
#     )  # 0-255にクリップし、uint8に変換
#     ax.imshow(img_restored)
#     ax.set_title(title, fontsize=10)
#     ax.axis("off")


# # ----- STEP 3: 図の表示 -----
# fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))  # figsizeを少し調整

# # --- サブプロット1: 元画像の表示 ---
# title_original = f"Original Image (Index: {index_to_analyze})"
# if x_sample_reshaped_preprocessed.shape[-1] == 3:  # RGB
#     display_denormalized_vgg16_for_subplot(
#         axes[0],
#         x_sample_reshaped_preprocessed,
#         title_original + "\n(VGG16 De-normalized)",
#     )
# elif x_sample_reshaped_preprocessed.shape[-1] == 1:  # Grayscale
#     img_min_orig = x_sample_reshaped_preprocessed.min()
#     img_max_orig = x_sample_reshaped_preprocessed.max()
#     epsilon_orig = 1e-6
#     if (img_max_orig - img_min_orig) > epsilon_orig:
#         scaled_image_orig = (x_sample_reshaped_preprocessed - img_min_orig) / (
#             img_max_orig - img_min_orig + epsilon_orig
#         )
#     else:
#         scaled_image_orig = np.zeros_like(x_sample_reshaped_preprocessed)
#     scaled_image_orig = np.clip(scaled_image_orig, 0, 1)
#     axes[0].imshow(scaled_image_orig.squeeze(), cmap="gray")
#     axes[0].set_title(title_original + "\n(Grayscale Scaled)", fontsize=10)
#     axes[0].axis("off")
# else:
#     axes[0].text(
#         0.5,
#         0.5,
#         "Cannot display original image\n(Unsupported channel num)",
#         ha="center",
#         va="center",
#         fontsize=9,
#     )
#     axes[0].set_title(title_original, fontsize=10)
#     axes[0].axis("off")

# # --- サブプロット2: 正規化されたヒートマップのみを表示 ---
# im = axes[1].imshow(
#     importance_heatmap_normalized,
#     cmap="inferno",
#     interpolation="bilinear",
#     vmin=0,
#     vmax=1,
# )
# axes[1].set_title(
#     f"Normalized Importance Heatmap\n(Index: {index_to_analyze})", fontsize=10
# )
# axes[1].axis("off")

# # カラーバーを追加
# fig.colorbar(
#     im,
#     ax=axes[1],
#     orientation="vertical",
#     fraction=0.046,
#     pad=0.04,
#     label="Normalized Importance (0-1)",
# )

# plt.suptitle(
#     f"Local Feature Importance Analysis for Sample {index_to_analyze}", fontsize=14
# )  # 全体のタイトル
# plt.tight_layout(rect=[0, 0, 1, 0.96])  # rectでsuptitleとの重なりを調整
# plt.show()


## 100個ランダムにLocalを保存

In [ ]:
import os

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf  # VGG16の逆前処理用 (もしダミーデータ生成時に使うなら)

# ----- STEP 0: 必要な変数の定義と設定 -----
# X_loaded, Y_predicted_loaded, Y_true_loaded は既にロード済みと仮定

IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 3

output_directory = "/home/kosukeyano/TUT_DTC Dropbox/Yano Kosuke/AIME/EM/Local_Feature_Importance_final"  # ディレクトリ名を適宜変更
os.makedirs(output_directory, exist_ok=True)

# # AIME_GPU オブジェクトと関連変数の準備 (実際のコードに合わせてください)
# # この部分は、あなたの環境で AIME_GPU をどのように初期化・使用しているかに依存します。
# try:
#     if "aime_gpu" not in locals() or aime_gpu is None:
#         raise NameError(
#             "'aime_gpu' object is not defined. Please initialize it and call 'create_explainer' first."
#         )
#     if "feature_names" not in locals() or feature_names is None:
#         # feature_names のダミー (ピクセル数に合わせたもの)
#         feature_names = [
#             f"pixel_{i}"
#             for i in range(
#                 IMG_HEIGHT * IMG_WIDTH * CHANNELS
#                 if CHANNELS > 1
#                 else IMG_HEIGHT * IMG_WIDTH
#             )
#         ]
#         print(
#             f"Warning: 'feature_names' not found. Using dummy feature_names with length {len(feature_names)}."
#         )
#     if "scaler" not in locals():  # scaler は使わない場合もあるので None でも許容
#         scaler = None
# except NameError as e:
#     print(f"Initialization Error: {e}")
#     raise

# # ロード済み変数の存在確認
# if not all(
#     var in locals() and locals()[var] is not None
#     for var in ["X_loaded", "Y_predicted_loaded", "Y_true_loaded"]
# ):
#     print(
#         "Error: 'X_loaded', 'Y_predicted_loaded', or 'Y_true_loaded' not found or are None. Please load them from .npz file."
#     )
#     raise NameError("Required data arrays are not loaded.")

# if not (
#     isinstance(X_loaded, np.ndarray)
#     and isinstance(Y_predicted_loaded, np.ndarray)
#     and isinstance(Y_true_loaded, np.ndarray)
# ):
#     print("Error: One or more loaded data variables are not NumPy arrays.")
#     raise TypeError(
#         "X_loaded, Y_predicted_loaded, or Y_true_loaded is not a NumPy array."
#     )

# if not (X_loaded.shape[0] == Y_predicted_loaded.shape[0] == Y_true_loaded.shape[0]):
#     print(
#         "Error: Mismatch in the number of samples between X_loaded, Y_predicted_loaded, and Y_true_loaded."
#     )
#     print(
#         f"Shapes: X_loaded={X_loaded.shape}, Y_predicted_loaded={Y_predicted_loaded.shape}, Y_true_loaded={Y_true_loaded.shape}"
#     )
#     raise ValueError("Sample count mismatch in loaded data.")


# ----- STEP 2 (ループ外に移動): 表示用関数の定義 (変更なし) -----
def display_denormalized_vgg16_for_subplot(ax, img_data_reshaped_preprocessed, title):
    if not (
        img_data_reshaped_preprocessed.ndim == 3
        and img_data_reshaped_preprocessed.shape[-1] == 3
    ):
        ax.text(
            0.5,
            0.5,
            f"Cannot display original image:\nExpected 3D RGB image,\ngot shape {img_data_reshaped_preprocessed.shape}",
            ha="center",
            va="center",
            fontsize=9,
        )
        ax.set_title(title, fontsize=10)
        ax.axis("off")
        return False
    img_restored = img_data_reshaped_preprocessed.copy()
    img_restored[..., 0] += 103.939
    img_restored[..., 1] += 116.779
    img_restored[..., 2] += 123.68
    img_restored = img_restored[..., ::-1]
    img_restored = np.clip(img_restored, 0, 255).astype(np.uint8)
    ax.imshow(img_restored)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return True


# ----- ループ処理 -----
start_index = 1
end_index = 100  # 1から100まで (100も含む)

print(
    f"\nStarting to generate and save images for indices {start_index} to {end_index}...\n"
)

for index_to_analyze in range(start_index, end_index + 1):
    print(f"--- Processing Sample Index: {index_to_analyze} ---")

    if not (0 <= index_to_analyze < X_loaded.shape[0]):
        print(
            f"Skipping: Index {index_to_analyze} is out of range for X_loaded (max index: {X_loaded.shape[0]-1})."
        )
        continue

    try:
        # ----- STEP 1 (ループ内): 分析対象のデータ準備 -----
        x_sample_flat_preprocessed = X_loaded[index_to_analyze]
        actual_value = Y_true_loaded[
            index_to_analyze
        ]  # ★★★ 正解ラベルを Y_true_loaded から取得 ★★★
        predicted_value = Y_predicted_loaded[
            index_to_analyze
        ]  # ★★★ 予測値を Y_predicted_loaded から取得 ★★★

        x_sample_cp = cp.asarray(x_sample_flat_preprocessed)
        # AIMEのy引数: 予測値に対する重要度を見る
        y_for_aime_cp = cp.array([predicted_value])  # または cp.array([actual_value])

        current_df_local_importance = aime_gpu.local_feature_importance_without_viz(
            x_sample_cp,
            y_for_aime_cp,
            feature_names=feature_names,
            scale=True,  # または False
            scaler=scaler,
            ignore_zero_features=True,
        )
        print(
            f"Local feature importance calculated. Actual: {actual_value:.4f}, Predicted: {predicted_value:.4f}"
        )

        # 画像データのリシェイプ
        target_size = IMG_HEIGHT * IMG_WIDTH * CHANNELS
        if CHANNELS == 1 and x_sample_flat_preprocessed.size == IMG_HEIGHT * IMG_WIDTH:
            target_size = IMG_HEIGHT * IMG_WIDTH
            x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
                IMG_HEIGHT, IMG_WIDTH, 1
            )
        elif x_sample_flat_preprocessed.size == target_size:
            x_sample_reshaped_preprocessed = x_sample_flat_preprocessed.reshape(
                IMG_HEIGHT, IMG_WIDTH, CHANNELS
            )
        else:
            raise ValueError(
                f"Flat image size {x_sample_flat_preprocessed.size} does not match dimensions."
            )

        # 特徴重要度の取得とリシェイプ
        if isinstance(current_df_local_importance, pd.DataFrame):
            local_importance_values_flat = current_df_local_importance.iloc[
                0
            ].to_numpy()
        elif isinstance(current_df_local_importance, cp.ndarray):
            local_importance_values_flat = cp.asnumpy(
                current_df_local_importance.flatten()
            )
        elif isinstance(current_df_local_importance, np.ndarray):
            local_importance_values_flat = current_df_local_importance.flatten()
        else:
            raise TypeError("current_df_local_importance is not of a recognized type.")

        if local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH:
            importance_heatmap_2d = local_importance_values_flat.reshape(
                IMG_HEIGHT, IMG_WIDTH
            )
        elif local_importance_values_flat.size == IMG_HEIGHT * IMG_WIDTH * CHANNELS:
            temp_heatmap_3d = local_importance_values_flat.reshape(
                IMG_HEIGHT, IMG_WIDTH, CHANNELS
            )
            importance_heatmap_2d = np.mean(temp_heatmap_3d, axis=2)
        else:
            raise ValueError(
                "Size of local_importance_values_flat does not match expected image dimensions."
            )

        # 特徴重要度の正規化
        all_importance_values_for_norm = importance_heatmap_2d.flatten()
        local_min = all_importance_values_for_norm.min()
        local_max = all_importance_values_for_norm.max()
        if (local_max - local_min) > 1e-6:
            importance_heatmap_normalized = (importance_heatmap_2d - local_min) / (
                local_max - local_min
            )
        else:
            importance_heatmap_normalized = np.zeros_like(importance_heatmap_2d)
        importance_heatmap_normalized = np.clip(importance_heatmap_normalized, 0, 1)

        # ----- STEP 3 (ループ内): 図の表示と保存 -----
        fig, axes = plt.subplots(1, 2, figsize=(12, 6.2))

        # サブプロット1: 元画像の表示
        title_original = f"Original Image (Index: {index_to_analyze})"
        original_displayed = False
        if x_sample_reshaped_preprocessed.shape[-1] == 3:
            original_displayed = display_denormalized_vgg16_for_subplot(
                axes[0],
                x_sample_reshaped_preprocessed,
                title_original,
            )
        elif x_sample_reshaped_preprocessed.shape[-1] == 1:
            img_min_orig, img_max_orig, epsilon_orig = (
                x_sample_reshaped_preprocessed.min(),
                x_sample_reshaped_preprocessed.max(),
                1e-6,
            )
            scaled_image_orig = (
                (x_sample_reshaped_preprocessed - img_min_orig)
                / (img_max_orig - img_min_orig + epsilon_orig)
                if (img_max_orig - img_min_orig) > epsilon_orig
                else np.zeros_like(x_sample_reshaped_preprocessed)
            )
            scaled_image_orig = np.clip(scaled_image_orig, 0, 1)
            axes[0].imshow(scaled_image_orig.squeeze(), cmap="gray")
            axes[0].set_title(title_original, fontsize=10)
            axes[0].axis("off")
            original_displayed = True
        else:
            axes[0].text(
                0.5,
                0.5,
                "Cannot display original image\n(Unsupported channel num)",
                ha="center",
                va="center",
                fontsize=9,
            )
            axes[0].set_title(title_original, fontsize=10)
            axes[0].axis("off")

        # サブプロット1の下に 正解値と予測値を表示
        axes[0].text(
            0.5,
            -0.08,
            f"Actual Value (True): {actual_value:.4f}\nPredicted Value: {predicted_value:.4f}",
            size=10,
            ha="center",
            transform=axes[0].transAxes,
            bbox=dict(boxstyle="round,pad=0.3", fc="aliceblue", alpha=0.9),
        )

        # サブプロット2: 正規化されたヒートマップのみを表示
        im = axes[1].imshow(
            importance_heatmap_normalized,
            cmap="inferno",
            interpolation="bilinear",
            vmin=0,
            vmax=1,
        )
        axes[1].set_title(
            f"Normalized Importance Heatmap\n(Index: {index_to_analyze})", fontsize=10
        )
        axes[1].axis("off")
        fig.colorbar(
            im,
            ax=axes[1],
            orientation="vertical",
            fraction=0.046,
            pad=0.04,
            label="Normalized Importance (0-1)",
        )

        fig.suptitle(
            f"Local Feature Importance - Sample {index_to_analyze}", fontsize=14
        )
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        save_filename = f"heatmap_sample_{index_to_analyze:03d}.png"
        full_save_path = os.path.join(output_directory, save_filename)
        plt.savefig(full_save_path, bbox_inches="tight")
        print(f"Saved: {full_save_path}")
        plt.close(fig)

    except Exception as e:
        print(f"Error processing sample index {index_to_analyze}: {e}")
        # エラーが発生した場合の処理 (例: ログに記録、ループ継続など)
        # import traceback
        # traceback.print_exc() # 詳細なエラー情報を表示する場合

print("\n--- All specified samples processed. ---")
